### 2D Integrals with Taylor expansions

Any definite and proper 2-dimensional integral can be discretized as shown:

$$
    \int_{c}^{d} \int_{a}^{b} \, f(x, y) \, dxdy = \sum_{i = 0}^{N_x - 1} \sum_{j = 0}^{N_y - 1} \, \int_{y_j}^{y_j + h_y} \int_{x_i}^{x_i + h_x} \, f(x, y) \, dxdy
$$

With $x_0 = a$, $x_{Nx} = b$, $h_x = \frac{b - a}{N_x}$ and $y_0 = c$, $y_{Ny} = d$, $h_y = \frac{d - c}{N_y}$

Trick: expand the function $f(x, y)$ in a Taylor series about the point $ \left( x_i + \frac{h_x}{2}, y_j + \frac{h_y}{2} \right) $ and integrate in $\left[ x_i, x_i + h_x \right] \times \left[ y_j, y_j + h_y \right]$

Using Maxima CAS we can write this approximation formula:

$$
    \int_{c}^{d} \int_{a}^{b} \, f(x, y) \, dxdy \approx h_x h_y \, \sum_{i = 0}^{N_x - 1} \sum_{j = 0}^{N_y - 1} \, f \left( x_i + \frac{h_x}{2}, y_j + \frac{h_y}{2} \right)
$$

The upper bound of the truncation error is given by:

$$
    E_{pes} \leq \frac{(b - a)(d - c)}{24} \left( h_x^2 M_{xx} + h_y^2 M_{yy} \right)
$$

Where $M_{xx} = \text{max} \left( | f_{xx}(x, y) | \right)$ and $M_{yy} = \text{max} \left( | f_{yy}(x, y) | \right)$ in $(x, y) \in [a, b] \times [c, d]$

Note: this upper bound is actually pessimistic! For a more optimistic error bound:

$$
    E_{opt} \leq \frac{1}{24} \left( h_x^2 \, \int_{c}^{d} \int_{a}^{b} \, \left| f_{xx}(x, y) \right| \, dxdy + h_y^2 \, \int_{c}^{d} \int_{a}^{b} \, \left| f_{yy}(x, y) \right| \, dxdy \right)
$$

In [1]:
import numpy as np
from scipy.integrate import dblquad # 2D integral method from SciPy

In [2]:
f = lambda x, y: np.exp(- x ** 2 - y ** 4) # f(x, y) = e^{-x^2 - y^4}
df_dx = lambda x, y: - 2.0 * x * f(x, y) # derivative of f(x, y) with respect to x
df_dy = lambda x, y: - 4.0 * y ** 3 * f(x, y) # derivative of f(x, y) with respect to y

In [3]:
def integral_2D(f, ab, cd, hx, hy): # Implements the formula above

    # Ranges, subdivisions and steps
    a, b = ab
    c, d = cd
    Nx = round((b - a) / hx)
    Ny = round((d - c) / hy)
    hx = (b - a) / Nx
    hy = (d - c) / Ny

    # Midpoints
    xs = a + (np.arange(Nx) + 0.5) * hx
    ys = c + (np.arange(Ny) + 0.5) * hy

    X, Y = np.meshgrid(xs, ys, indexing = "ij") # Meshgrid

    I = hx * hy * np.sum(f(X, Y)) # 2D integral result

    return I # Integral result

In [4]:
ab = (- 3.0, 3.0) # a = - 3, b = 3
cd = (- 3.0, 3.0) # c = - 3, d = 3
hx, hy = 0.005, 0.005 #  Same step

res = integral_2D(f, ab, cd, hx, hy) # Result
res

np.float64(3.213042145385196)

In [5]:
# Computing the optimistic error bound
int1 = integral_2D(
    f = lambda x, y: np.abs(df_dx(x, y)),
    ab = ab,
    cd = cd,
    hx = hx,
    hy = hy
)

int2 = integral_2D(
    f = lambda x, y: np.abs(df_dy(x, y)),
    ab = ab,
    cd = cd,
    hx = hx,
    hy = hy
)

err_opt = 1 / 24.0 * ((hx ** 2) * int1 + (hy ** 2) * int2)
err_opt

np.float64(7.468749414276554e-06)

In [6]:
# Computing the pessimistic error upper bound: M_xx = M_yy = 1 in our case
err_pes = ((ab[1] - ab[0]) * (cd[1] - cd[0]) / 24.0) * (hx ** 2 + hy ** 2)
err_pes

7.500000000000001e-05

In [7]:
res_scipy = dblquad(f, ab[0], ab[1], cd[0], cd[1]) # from Scipy
res_scipy

(3.213042142588783, 2.784422957827215e-08)

In [8]:
# Full comparison
print(f"res_opt = {res:.6f} +- {err_opt:.6f}")
print(f"res_pes = {res:.5f} +- {err_pes:.5f}")
print(f"res_scipy = {res_scipy[0]:.8f} +- {res_scipy[1]:.8f}")

res_opt = 3.213042 +- 0.000007
res_pes = 3.21304 +- 0.00008
res_scipy = 3.21304214 +- 0.00000003
